# 5 · Ingesting Data
*Intro to Python for Scientists & Public Health Professionals*

Real analysis starts with getting data *in*. `read_csv` is the workhorse, and it has many options for the messy realities of real files — no header row, columns you don't want, types you need to force, dates to parse. We'll also read JSON and pull live data from a public-health API.

### By the end of this notebook you can
- Read a CSV with and without a header row
- Control a read with `names`, `usecols`, `index_col`, `dtype`, `parse_dates`, and `nrows`
- Read JSON and pull data from a REST API into a DataFrame
- Rename columns after loading

### Agenda
1. Reading a CSV
2. `read_csv` options
3. Other sources: JSON, Excel, APIs
4. Renaming columns
5. Writing data back out

### How the exercises work
Each exercise has a prompt, an empty cell to try it yourself, and a collapsed **Solution** you can expand to check your work.

In [ ]:
import numpy as np
import pandas as pd

BASE_URL = "https://raw.githubusercontent.com/jimcody2014/2026-python-data/refs/heads/main"

## 1. Reading a CSV

The simple case: a file whose first row holds the column names. `diabetes1.csv` is exactly that.

In [ ]:
diabetes1 = pd.read_csv(f"{BASE_URL}/diabetes1.csv")
print(diabetes1.shape)
diabetes1.head()

> Reference: [`read_csv` options](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) — there are many; we'll use the common ones.

## 2. `read_csv` options

`diabetes2.csv` is the *same data with no header row*. Read naively, pandas treats the first data row as headers — so we need to tell it otherwise.

In [ ]:
# header=None: don't treat any row as names; pandas numbers the columns 0..n
raw = pd.read_csv(f"{BASE_URL}/diabetes2.csv", header=None)
raw.head()

Supply real names with `names=`.

In [ ]:
names = ["encounter_id", "patient_nbr", "race", "gender", "age", "weight",
         "admission_type_id", "admission_source_id", "time_in_hospital", "payer_code",
         "num_lab_procedures", "num_procedures", "num_medications", "diag_1", "A1Cresult",
         "metformin", "miglitol", "insulin", "readmitted", "date"]

df = pd.read_csv(f"{BASE_URL}/diabetes2.csv", header=None, names=names)
df.head()

Parse the date column as real dates, and set a column as the index — both at read time.

In [ ]:
df = pd.read_csv(f"{BASE_URL}/diabetes2.csv", header=None, names=names,
                 parse_dates=["date"],
                 index_col="encounter_id")
df[["age", "time_in_hospital", "date"]].head()

> Date note: these values look like `10/20/21`. pandas guesses the format; when it's ambiguous (is `03/04` March 4th or April 3rd?), pass an explicit `format=` or `dayfirst=`. We'll go deeper in the datetime lesson.

Keep only the columns you need with `usecols`, and read just the first rows of a large file with `nrows` — handy for a quick look before committing to a 100k-row load.

In [ ]:
df = pd.read_csv(f"{BASE_URL}/diabetes2.csv", header=None, names=names,
                 usecols=["encounter_id", "age", "time_in_hospital", "date"],
                 parse_dates=["date"],
                 index_col="encounter_id",
                 nrows=1000)
print(df.shape)
df.head()

Force a column's type at read time with `dtype` — for example, keep an ID as text so leading zeros and formatting survive.

In [ ]:
df = pd.read_csv(f"{BASE_URL}/diabetes2.csv", header=None, names=names,
                 usecols=["encounter_id", "patient_nbr", "age"],
                 dtype={"patient_nbr": "string"})
df.dtypes

## 3. Other sources: JSON, Excel, APIs

### JSON
The immunotherapy dataset is hosted as JSON. `read_json` handles a URL the same way `read_csv` does.

In [ ]:
imm = pd.read_json(f"{BASE_URL}/immjson.json")
print(imm.shape)
imm.head()

### Excel
`read_excel` works much like `read_csv` (with `sheet_name` to pick a tab), but it needs a real file and the `openpyxl` engine. The cell below is intentionally **not runnable** — and that's the teaching point.

In [ ]:
# Excel is NOT read here on purpose:
#   imm = pd.read_excel("immunotherapy.xlsx", sheet_name="Immuno1")
#
# A binary .xlsx does not serve over a raw GitHub link the way a CSV does, and a path
# like "/Users/.../immunotherapy.xlsx" only exists on one person's laptop. That is exactly
# why this course keeps its data as CSV/JSON in a public repo — it travels.
# (The same immunotherapy data is available as JSON in the cell above.)
print("read_excel demo is discussed, not executed — see the comment above.")

### A live API
Many public-health sources (CDC, WHO, Census) expose REST APIs that return JSON. The pattern is always: `requests.get(url)` → `.json()` → `DataFrame`. Because it depends on a live service, we wrap it in `try`/`except` so a momentary outage or rate-limit doesn't halt the notebook.

In [ ]:
import requests

url = "https://data.cdc.gov/resource/saz5-9hgg.json"
try:
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    vaccines = pd.DataFrame(response.json())
    print("rows, cols:", vaccines.shape)
except Exception as e:
    vaccines = pd.DataFrame()
    print(f"Live API didn't return data this run ({type(e).__name__}). "
          "Public APIs can rate-limit or be briefly unavailable — just re-run. "
          "The pattern is still: requests.get(url) -> .json() -> pd.DataFrame.")

In [ ]:
vaccines.head()

### Exercise — Read with options *(15 min)*

**Part 1.** Read `diabetes_inspect.csv` into a DataFrame named `df` and report its shape.

**Part 2.** Read `diabetes2.csv` (no header row) but bring in **only** the columns for encounter id, age, time in hospital, and date. Use the encounter id as the index, parse the date as a real date, and report the shape and dtypes.

In [ ]:
# Your work here


<details>
<summary>Solution</summary>

```python
# Part 1
df = pd.read_csv(f"{BASE_URL}/diabetes_inspect.csv")
print(df.shape)

# Part 2
names = ["encounter_id", "patient_nbr", "race", "gender", "age", "weight",
         "admission_type_id", "admission_source_id", "time_in_hospital", "payer_code",
         "num_lab_procedures", "num_procedures", "num_medications", "diag_1", "A1Cresult",
         "metformin", "miglitol", "insulin", "readmitted", "date"]

dia = pd.read_csv(f"{BASE_URL}/diabetes2.csv", header=None, names=names,
                  usecols=["encounter_id", "age", "time_in_hospital", "date"],
                  index_col="encounter_id",
                  parse_dates=["date"])
print(dia.shape)
print(dia.dtypes)
```

**Why this works.** Part 1 is the default case — a header row, nothing special. Part 2 stacks the options: `header=None` plus `names` because there's no header to read; `usecols` to keep only four columns (named, so it's readable); `index_col` to promote encounter id out of the columns and into the row index; and `parse_dates` so `date` arrives as a real datetime rather than text. Doing this at read time is cleaner than loading everything and fixing it afterward.

</details>

## 4. Renaming columns

Often the first thing you do after loading. Three idioms cover almost everything (we'll do more tidying in the next lesson).

In [ ]:
imm = pd.read_json(f"{BASE_URL}/immjson.json")
print("original:", imm.columns.tolist())

In [ ]:
# (a) rename specific columns by mapping old -> new
imm = imm.rename(columns={"Number_of_Warts": "n_warts", "Result_of_Treatment": "outcome"})
print("after rename:", imm.columns.tolist())

In [ ]:
# (b) replace ALL names at once by assigning to .columns
#     (must match the number of columns)
imm.columns = ["sex", "age", "time", "n_warts", "type", "area", "induration", "outcome"]
print("after reset:", imm.columns.tolist())

In [ ]:
# (c) transform names with vectorized string methods (great for bulk fixes)
imm.columns = imm.columns.str.upper()
print("after str.upper:", imm.columns.tolist())

> Note: older code uses `rename(..., inplace=True)` or `set_axis(..., inplace=True)`. The `inplace` argument on `set_axis` was removed in pandas 2.0 — prefer assigning the result back, e.g. `df = df.set_axis(new_names, axis=1)`.

## 5. Writing data back out

The counterpart to reading: once you've cleaned or summarized data, save it so others — or future-you — can use it without redoing the work.

In [ ]:
imm.to_csv("immunotherapy_clean.csv", index=False)   # index=False: don't write row numbers as a column
print("saved immunotherapy_clean.csv")

> Excel works the same way: `imm.to_excel("out.xlsx", index=False)` (needs the `openpyxl` engine). In Colab, files you write show up in the file browser on the left, where you can download them.

## Wrap-up

You can read CSVs with full control over headers, columns, types, and dates; pull from JSON and live APIs; rename columns on arrival; and write results back out.

**Next:** Indexes — selecting data precisely with `loc` and `iloc`.